# Diabetic Retinopathy Grading on APTOS 2019: reproducing Chilukoti et al. (2024)

**Task.** Look at a retina photo (fundus image) and grade diabetic retinopathy (DR) on a 5-step scale:
`0 No DR · 1 Mild · 2 Moderate · 3 Severe · 4 Proliferative (PDR)`.

**Paper being reproduced.** *A reliable diabetic retinopathy grading via transfer learning and ensemble learning with quadratic weighted kappa metric* (BMC Med Inform Decis Mak, 2024). It reports **QWK 0.954** on APTOS 2019. The authors released no code, so everything here is re-implemented from the paper text.

**What this notebook does**

| Step | What happens |
|---|---|
| 1 | Load the APTOS 2019 images and labels straight from Kaggle's input folder |
| 2 | Split 70 / 15 / 15 (train / val / test), keeping the class balance in each part |
| 3 | Pre-process every image once: CLAHE contrast boost, light blur, resize to 300×300 |
| 4 | Fine-tune an ImageNet-pretrained **EfficientNet-B3** with the paper's 3-layer head |
| 5 | Keep predictions from 10 evenly spaced epochs and combine them by majority vote (the "snapshot ensemble") |
| 6 | Run it twice: with the paper's learning rate (1e-3) and with 1e-4, then compare |

**Metric.** QWK (quadratic weighted kappa) is the headline number. It punishes a prediction more the further it is from the true grade (calling Severe "Moderate" costs little, calling it "No DR" costs a lot). 1.0 is perfect, 0 is chance.

---

### How to run this on Kaggle (free GPU)

1. On kaggle.com go to **Code → New Notebook → File → Import Notebook** and upload this file.
2. **Add the data:** right sidebar **Input → Add Input → Competitions**, search *APTOS 2019 Blindness Detection* and add it. You must have joined the competition once (open its page and accept the rules).
3. **Turn on the GPU:** sidebar **Session options → Accelerator → GPU T4 ×2** (a P100 also works).
4. **Turn on Internet:** sidebar **Session options → Internet → On**. It is needed once, to download the ImageNet weights. (Kaggle asks for phone verification the first time.)
5. Click **Run All**. For a long run you can instead choose **Save Version → Save & Run All (Commit)**. That runs in the background, so you can close the tab.

> Tip: set `QUICK_TEST = True` in the next cell for a 2-minute check that everything is wired up, then set it back to `False`.

## 1. Setup and settings

In [ ]:
import hashlib
import json
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    import timm
except ImportError:                      # not preinstalled on this machine
    !pip install -q timm
    import timm

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "PDR"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

ON_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU found (very slow!)")

In [ ]:
QUICK_TEST = False   # True = tiny run (few epochs, few batches) just to check the pipeline works


@dataclass
class Config:
    # Values below are the paper's setup (Chilukoti et al. 2024, strategy 2).
    image_size: int = 300
    epochs: int = 60
    batch_size: int = 16
    lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip: float = 0.1        # clip the global gradient norm to this value
    n_snapshots: int = 10         # predictions kept for the ensemble, evenly spaced over training
    seed: int = 42
    backbone: str = "efficientnet_b3"
    pretrained: bool = True       # start from ImageNet weights
    use_amp: bool = True          # mixed precision: faster on GPU, same idea
    max_train_batches: int = 0    # 0 = use all batches; only >0 for the quick test


BASE_CFG = Config()
if QUICK_TEST:
    BASE_CFG = replace(BASE_CFG, epochs=4, n_snapshots=2, max_train_batches=5, image_size=224)
if device.type != "cuda":
    BASE_CFG = replace(BASE_CFG, use_amp=False)    # mixed precision is a GPU feature

# The two experiments. Only the learning rate differs.
EXPERIMENTS = {
    "paper_lr1e-3": replace(BASE_CFG, lr=1e-3),
    "lr1e-4":       replace(BASE_CFG, lr=1e-4),
}
print(json.dumps(asdict(BASE_CFG), indent=2))

## 2. Load the data

APTOS 2019 has 3,662 labelled retina photos. The classes are very unbalanced: about half are "No DR", while Severe is only ~5%. That imbalance is the root of most of the problems we will see later.

In [ ]:
def find_aptos_dir() -> Path:
    # Find the folder holding train.csv + train_images/. Set APTOS_DIR yourself to run outside Kaggle.
    if os.environ.get("APTOS_DIR"):
        return Path(os.environ["APTOS_DIR"])
    for pattern in ("*", "*/*", "*/*/*"):                       # Kaggle mounts differ between UI versions
        for folder in Path("/kaggle/input").glob(pattern):
            if (folder / "train.csv").exists() and (folder / "train_images").exists():
                return folder
    raise FileNotFoundError("APTOS data not found. Did you add the competition under 'Input' in the sidebar?")


DATA_DIR = find_aptos_dir()
df = pd.read_csv(DATA_DIR / "train.csv").rename(columns={"diagnosis": "label"})
df["path"] = df["id_code"].map(lambda i: DATA_DIR / "train_images" / f"{i}.png")
print(f"data folder: {DATA_DIR}\n{len(df)} images")

counts = df["label"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(CLASS_NAMES, counts.values, color="#4C78A8")
for i, c in enumerate(counts.values):
    ax.text(i, c + 15, f"{c} ({c / len(df):.0%})", ha="center", fontsize=9)
ax.set(title="Class distribution (APTOS 2019, training set)", ylabel="images")
plt.tight_layout(); plt.show()

### Optional check: duplicate images

Earlier work found that APTOS contains groups of **byte-identical images**, some of them with *conflicting* grades. A random split can then put a copy of a test image into the training set. Earlier work measured the effect on QWK as only about 0.001, so we just look and move on. This cell takes about a minute.

In [ ]:
def file_hash(path):
    return hashlib.md5(path.read_bytes()).hexdigest()

with ThreadPoolExecutor(os.cpu_count()) as pool:
    df["hash"] = list(tqdm(pool.map(file_hash, df["path"]), total=len(df), desc="hashing"))

groups = df.groupby("hash")["label"].agg(["size", "nunique"])
groups = groups[groups["size"] > 1]
print(f"{len(groups)} groups of identical images, {int((groups['nunique'] > 1).sum())} of them with conflicting grades")

## 3. Split into train / validation / test

* **train (70%)**: the model learns from these.
* **validation (15%)**: used during training to watch progress and pick the "best" epoch.
* **test (15%)**: never touched until the end. All reported numbers come from here.

`stratify` keeps the class proportions identical in all three parts, which matters because Severe and PDR are so rare. The seed makes the split repeatable.

In [ ]:
def make_split(df, seed=42, val_frac=0.15, test_frac=0.15):
    # Returns row positions (into df) for the train, val and test sets.
    positions = np.arange(len(df))
    train_val, test = train_test_split(positions, test_size=test_frac, stratify=df["label"], random_state=seed)
    train, val = train_test_split(train_val, test_size=val_frac / (1 - test_frac),
                                  stratify=df["label"].to_numpy()[train_val], random_state=seed)
    return {"train": np.sort(train), "val": np.sort(val), "test": np.sort(test)}


labels = df["label"].to_numpy()
split = make_split(df, BASE_CFG.seed)
pd.DataFrame({name: np.bincount(labels[idx], minlength=NUM_CLASSES) for name, idx in split.items()},
             index=CLASS_NAMES)

## 4. Pre-processing (once, up front)

The paper's recipe: boost local contrast with **CLAHE**, smooth with a small **Gaussian blur**, resize to a square.

* CLAHE is applied only to the *lightness* channel (LAB colour space), so colours are not distorted.
* The raw photos are huge (up to 4000×3000), so each one is first shrunk to at most 512 px on the long side. That keeps the CLAHE step fast.
* The result is the same every time, so we compute it once for all images and keep them in memory (about 1 GB at 300×300).

In [ ]:
def preprocess(path, size, enhance=True):
    # Load -> shrink -> (CLAHE + blur) -> square resize. Returns an RGB uint8 array (size, size, 3).
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    h, w = bgr.shape[:2]
    if max(h, w) > 512:
        scale = 512 / max(h, w)
        bgr = cv2.resize(bgr, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
    if enhance:
        lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
        lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
        bgr = cv2.GaussianBlur(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), (5, 5), 0)
    bgr = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


# One example per grade: before (top) and after (bottom).
examples = [df[df["label"] == c].iloc[0]["path"] for c in range(NUM_CLASSES)]
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(14, 6))
for c, path in enumerate(examples):
    axes[0, c].imshow(preprocess(path, 300, enhance=False)); axes[0, c].set_title(CLASS_NAMES[c])
    axes[1, c].imshow(preprocess(path, 300, enhance=True))
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].text(-0.05, 0.5, "original", transform=axes[0, 0].transAxes, rotation=90, ha="right", va="center")
axes[1, 0].text(-0.05, 0.5, "CLAHE + blur", transform=axes[1, 0].transAxes, rotation=90, ha="right", va="center")
plt.tight_layout(); plt.show()

In [ ]:
def build_image_cache(paths, size):
    images = np.empty((len(paths), size, size, 3), dtype=np.uint8)

    def fill(i):
        images[i] = preprocess(paths[i], size)

    with ThreadPoolExecutor(os.cpu_count()) as pool:            # OpenCV releases the GIL, so threads are enough
        list(tqdm(pool.map(fill, range(len(paths))), total=len(paths), desc="pre-processing"))
    return images


t0 = time.time()
images = build_image_cache(df["path"].tolist(), BASE_CFG.image_size)
print(f"{images.shape}, {images.nbytes / 1e9:.2f} GB, {time.time() - t0:.0f}s")

## 5. Model

EfficientNet-B3 pretrained on ImageNet, with its original classifier removed and replaced by the paper's head:

```
1536 features -> Linear 512 -> Dropout 0.5 -> ReLU -> Linear 512 -> Dropout 0.25 -> ReLU -> Linear 5
```

All layers are fine-tuned (nothing is frozen). Training uses plain cross-entropy loss.

In [ ]:
class DRNet(nn.Module):
    def __init__(self, backbone="efficientnet_b3", pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="avg")
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 512), nn.Dropout(0.5), nn.ReLU(inplace=True),
            nn.Linear(512, 512), nn.Dropout(0.25), nn.ReLU(inplace=True),
            nn.Linear(512, NUM_CLASSES),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


_mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
_std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)


def to_model_input(batch_uint8):
    # numpy (B, H, W, 3) uint8 -> normalised float tensor (B, 3, H, W) on the GPU.
    x = torch.from_numpy(batch_uint8).to(device).permute(0, 3, 1, 2).float() / 255.0
    return ((x - _mean) / _std).contiguous(memory_format=torch.channels_last)

## 6. Metrics and the snapshot ensemble

Besides QWK we report accuracy, macro-F1, macro AUC and **per-class recall**. Per-class recall matters here: a model can reach a decent QWK while never predicting a rare grade at all.

**Snapshot ensemble (paper, strategy 2).** During training we save the model's predictions at 10 evenly spaced epochs. The final grade is the *most common* prediction among those 10 (a majority vote). Ties are broken by the average probability.

In [ ]:
def compute_metrics(y_true, y_pred, probs):
    labels_all = list(range(NUM_CLASSES))
    prec, rec, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels_all, zero_division=0)
    return {
        "qwk": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_auc": float(roc_auc_score(y_true, probs, multi_class="ovr", average="macro")),
        "recall": {CLASS_NAMES[c]: float(rec[c]) for c in labels_all},
        "minority_recall": float(np.mean([rec[3], rec[4]])),       # mean recall of Severe and PDR
        "off_by_2_or_more": int(np.sum(np.abs(y_true - y_pred) >= 2)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels_all).tolist(),
    }


def majority_vote(snapshot_probs):
    # snapshot_probs: (n_snapshots, n_images, 5) -> one predicted grade per image.
    votes = snapshot_probs.argmax(-1)                                          # each snapshot's vote
    tally = np.stack([(votes == c).sum(0) for c in range(NUM_CLASSES)], 1).astype(float)
    tally += 1e-3 * snapshot_probs.mean(0)                                     # tie-break
    return tally.argmax(1)

## 7. Training

Per epoch: one pass over the training set, then predict the validation set and log its QWK. We also store:

* **best-validation model**: test predictions from the epoch with the highest validation QWK (the usual "pick the best checkpoint" approach)
* **snapshots**: test predictions at the 10 evenly spaced epochs, for the ensemble

Nothing from the test set is used to make any decision. It is only predicted so we can report numbers at the end.

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict_probs(model, indices, cfg, batch_size=32):
    model.eval()
    out = []
    for start in range(0, len(indices), batch_size):
        x = to_model_input(images[indices[start:start + batch_size]])
        with torch.autocast(device.type, dtype=torch.float16, enabled=cfg.use_amp):
            logits = model(x)
        out.append(torch.softmax(logits.float(), dim=1).cpu())
    return torch.cat(out).numpy()


def train_one_epoch(model, optimizer, scaler, criterion, cfg, rng):
    model.train()
    order = rng.permutation(split["train"])
    n_batches = len(order) // cfg.batch_size                                   # drop the last partial batch
    if cfg.max_train_batches:
        n_batches = min(n_batches, cfg.max_train_batches)
    total_loss, seen = 0.0, 0
    for b in range(n_batches):
        idx = np.sort(order[b * cfg.batch_size:(b + 1) * cfg.batch_size])      # sorted = faster memory reads
        x = to_model_input(images[idx])
        y = torch.from_numpy(labels[idx]).to(device)

        with torch.autocast(device.type, dtype=torch.float16, enabled=cfg.use_amp):
            logits = model(x)
        loss = criterion(logits.float(), y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                                             # so clipping sees real gradient sizes
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(y)
        seen += len(y)
    return total_loss / seen

In [ ]:
def run_experiment(name, cfg):
    # Train one configuration; return its training log, test predictions and metrics.
    print(f"\n=== {name} ===  lr={cfg.lr}  epochs={cfg.epochs}  batch={cfg.batch_size}  size={cfg.image_size}")
    seed_everything(cfg.seed)
    model = DRNet(cfg.backbone, cfg.pretrained).to(device).to(memory_format=torch.channels_last)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    criterion = nn.CrossEntropyLoss()
    rng = np.random.default_rng(cfg.seed)

    snapshot_epochs = {round(cfg.epochs * k / cfg.n_snapshots) for k in range(1, cfg.n_snapshots + 1)}
    log, snapshot_test_probs = [], []
    best_val_qwk, best_test_probs, best_state = -2.0, None, None

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, optimizer, scaler, criterion, cfg, rng)

        val_probs = predict_probs(model, split["val"], cfg)
        val_pred = val_probs.argmax(1)
        val_qwk = cohen_kappa_score(labels[split["val"]], val_pred, weights="quadratic")
        log.append({"epoch": epoch, "train_loss": train_loss, "val_qwk": float(val_qwk),
                    "val_acc": float(accuracy_score(labels[split["val"]], val_pred))})
        print(f"epoch {epoch:2d}/{cfg.epochs} | loss {train_loss:.3f} | val QWK {val_qwk:.3f} | "
              f"val acc {log[-1]['val_acc']:.3f} | {time.time() - t0:.0f}s")

        if val_qwk > best_val_qwk:                                             # new best on validation
            best_val_qwk = val_qwk
            best_test_probs = predict_probs(model, split["test"], cfg)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if epoch in snapshot_epochs:                                           # ensemble member
            snapshot_test_probs.append(predict_probs(model, split["test"], cfg))

    y_test = labels[split["test"]]
    snaps = np.stack(snapshot_test_probs)
    results = {
        "best_val_single": compute_metrics(y_test, best_test_probs.argmax(1), best_test_probs),
        "last_epoch_single": compute_metrics(y_test, snaps[-1].argmax(1), snaps[-1]),
        "snapshot_vote": compute_metrics(y_test, majority_vote(snaps), snaps.mean(0)),
        "snapshot_mean": compute_metrics(y_test, snaps.mean(0).argmax(1), snaps.mean(0)),
    }

    run_dir = OUTPUT_DIR / "runs" / name                                       # save everything for later
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))
    (run_dir / "results.json").write_text(json.dumps(results, indent=2))
    pd.DataFrame(log).to_csv(run_dir / "log.csv", index=False)
    np.savez_compressed(run_dir / "test_probs.npz", best=best_test_probs, snapshots=snaps, y=y_test)
    torch.save(best_state, run_dir / "best_val_weights.pt")
    return {"log": pd.DataFrame(log), "results": results, "config": cfg}

## 8. Run the experiments

* **`paper_lr1e-3`** uses exactly the paper's hyper-parameters.
* **`lr1e-4`** changes *only* the learning rate. In the earlier local reproduction the paper's 1e-3 made training unstable (validation QWK repeatedly collapsed), and 1e-4 fixed most of it.

On a Kaggle T4 expect roughly 30-60 minutes per run at the full 60 epochs.

In [ ]:
runs = {}
for name, cfg in EXPERIMENTS.items():
    runs[name] = run_experiment(name, cfg)

## 9. Results

In [ ]:
PAPER_QWK = 0.954        # reported by the paper for APTOS 2019 with ImageNet initialisation

PREDICTION_LABELS = {"best_val_single": "single model (best val epoch)", "last_epoch_single": "single model (last epoch)",
                     "snapshot_vote": "10-snapshot majority vote", "snapshot_mean": "10-snapshot mean probability"}
rows = []
for name, run in runs.items():
    for key, label in PREDICTION_LABELS.items():
        m = run["results"][key]
        rows.append({"experiment": name, "prediction": label, "QWK": m["qwk"], "accuracy": m["accuracy"],
                     "macro-F1": m["macro_f1"], "macro AUC": m["macro_auc"],
                     "Severe recall": m["recall"]["Severe"], "PDR recall": m["recall"]["PDR"],
                     "errors >= 2 grades": m["off_by_2_or_more"]})
summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
print(f"Paper reports QWK {PAPER_QWK} (APTOS 2019, ImageNet init).  Test set: {len(split['test'])} images.\n")
summary.style.format(precision=3).background_gradient(subset=["QWK"], cmap="Greens").hide(axis="index")

### Training curves

If the validation QWK swings wildly from epoch to epoch (or falls to zero), the learning rate is too high for stable training.

In [ ]:
fig, axes = plt.subplots(1, len(runs), figsize=(6.5 * len(runs), 3.8), squeeze=False)
for ax, (name, run) in zip(axes[0], runs.items()):
    log = run["log"]
    ax.plot(log["epoch"], log["val_qwk"], label="validation QWK", color="#4C78A8")
    ax.plot(log["epoch"], log["val_acc"], label="validation accuracy", color="#F58518", alpha=0.7)
    ax.set(title=f"{name}  (lr={run['config'].lr})", xlabel="epoch", ylim=(-0.05, 1.0))
    twin = ax.twinx()
    twin.plot(log["epoch"], log["train_loss"], color="gray", linestyle="--", label="train loss")
    twin.set_ylabel("train loss")
    ax.legend(loc="lower left"); twin.legend(loc="lower right")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150); plt.show()

### Confusion matrices (test set)

Rows are the true grade, columns the predicted grade. Numbers on the diagonal are correct. Look at the **Severe** row: if its diagonal cell is empty, the model has learned to never predict that grade, even when QWK still looks good.

In [ ]:
fig, axes = plt.subplots(len(runs), 2, figsize=(11, 4.6 * len(runs)), squeeze=False)
for r, (name, run) in enumerate(runs.items()):
    for c, key in enumerate(["best_val_single", "snapshot_vote"]):
        m = run["results"][key]
        cm = np.array(m["confusion_matrix"])
        ax = axes[r, c]
        ax.imshow(cm / cm.sum(1, keepdims=True).clip(1), cmap="Blues", vmin=0, vmax=1)    # shading = per-row share
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                ax.text(j, i, cm[i, j], ha="center", va="center",
                        color="white" if cm[i, j] / max(cm[i].sum(), 1) > 0.5 else "black")
        ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES), xlabel="predicted", ylabel="true",
               title=f"{name}: {PREDICTION_LABELS[key]}\nQWK {m['qwk']:.3f}")
        ax.set_xticklabels(CLASS_NAMES, rotation=30); ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=150); plt.show()

## 10. How to read the outcome

What the earlier local run (224 px, 30 epochs, Apple M2) found. Your Kaggle run uses the full paper setting, so numbers will differ somewhat. The *pattern* is what to compare:

| Observation (local run) | Why it matters |
|---|---|
| lr 1e-3 → QWK 0.83, far below the paper's 0.954 | The paper's hyper-parameters alone did not reproduce its result |
| Validation QWK collapsed and recovered during training at lr 1e-3 | Unstable training |
| Snapshot vote was *worse* than a single model at lr 1e-3 | Voting over mostly weak snapshots does not help |
| Severe grade never predicted at lr 1e-3, yet QWK stayed ~0.83 | QWK and accuracy hide failures on rare grades |
| lr 1e-4 → QWK 0.887, Severe recall rose from 0 to ~0.3 | A stable learning rate closes about half of the gap |

**Things to check in your results:** does 300 px / 60 epochs close more of the gap? Is Severe recall still near zero? Does the ensemble beat the single model?

**Next step (Stage 4):** replace cross-entropy with a class-balanced, ordinal-aware loss to help the rare grades. `nn.CrossEntropyLoss()` in `run_experiment` is the single line to change.

**Files written to `/kaggle/working`** (they appear under *Output* after a committed run): `summary.csv`, `training_curves.png`, `confusion_matrices.png`, and per run `runs/<name>/{config.json, results.json, log.csv, test_probs.npz, best_val_weights.pt}`.

*This is a reproduction study, not a clinical tool.*